# DreamBooth Reproduction on Stable Diffusion v1.5
**Paper:** Ruiz et al. 2022, arXiv:2208.12242

**Subjects:** dog (corgi), bear_plushie, colorful_sneaker, teapot

**Runs:** 11 total (4 main + 4 no_prior + 3 dog ablations)

---

## 1. Install + Imports + Drive Mount

In [ ]:
# Install dependencies (run once per session)
!pip -q install --upgrade diffusers transformers accelerate safetensors lpips ftfy timm
import torch
print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch 2.10.0+cu128, CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [ ]:
import os
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
else:
    print("Drive already mounted.")


Drive already mounted.


In [ ]:
import gc, json, math, csv, pickle, random, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
    StableDiffusionPipeline,
    DPMSolverMultistepScheduler,
)
from transformers import CLIPTokenizer, CLIPTextModel, CLIPModel, CLIPProcessor
import lpips

warnings.filterwarnings("ignore")


## 2. Configuration

In [ ]:
CFG = {
    # Paths
    "base_path": "/content/drive/MyDrive/FinalProjectDL",
    "output_path": "/content/drive/MyDrive/FinalProjectDL/final_run",
    "dataset_root": "/content/drive/MyDrive/FinalProjectDL/dreambooth_dataset/dataset",
    "class_images_dirname": "class_images_final",

    # Model
    "pretrained_model": "runwayml/stable-diffusion-v1-5",
    "identifier": "sks",
    "seed": 42,

    # Subjects
    "subjects": [
        {"name": "dog",              "class_noun": "dog",            "is_living": True},
        {"name": "bear_plushie",     "class_noun": "stuffed animal", "is_living": False},
        {"name": "colorful_sneaker", "class_noun": "sneaker",        "is_living": False},
        {"name": "teapot",           "class_noun": "teapot",         "is_living": False},
    ],
    "headline_subject": "dog",

    # Prompt templates
    "instance_prompt_template": "a photo of a {identifier} {class_noun}",
    "class_prompt_template": "a photo of a {class_noun}",

    # Class prior generation
    "num_class_images": 200,
    "class_gen_steps": 50,
    "class_gen_batch": 2,

    # Training
    "resolution": 512,
    "train_batch_size": 1,
    "max_train_steps": 800,
    "lr_unet": 2.5e-6,
    "lr_te": 2e-6,
    "lambda_prior": 1.0,
    "max_grad_norm": 1.0,

    # Inference
    "sample_steps": 50,
    "guidance_scale": 7.5,
    "images_per_prompt": 4,

    # Metrics
    "clip_model_name": "openai/clip-vit-base-patch32",
    "dino_repo": "facebookresearch/dino:main",
    "dino_model": "dino_vits16",

    # Appendix flags
    "RUN_MIXING_APPENDIX": True,
    "RUN_MIXING_VARIANT_2": True,
    "RUN_MIXING_VARIANT_3": True,
    "mixing_dogs": ["dog", "dog3"],

    # Final runtime release: off by default
    "RUN_FINAL_UNASSIGN": True,
}

# Derived paths
BASE = Path(CFG["base_path"])
DATASET_ROOT = Path(CFG["dataset_root"])
OUT = Path(CFG["output_path"])
CLASS_ROOT = OUT / CFG["class_images_dirname"]
VANILLA_ROOT = OUT / "vanilla_sd_baseline"
METRICS_CSV = OUT / "all_metrics.csv"
RESULTS_PKL = OUT / "all_results.pkl"

OUT.mkdir(parents=True, exist_ok=True)
CLASS_ROOT.mkdir(parents=True, exist_ok=True)
VANILLA_ROOT.mkdir(parents=True, exist_ok=True)

# Load previous results if they exist
ALL_RESULTS = {}
if RESULTS_PKL.exists():
    with open(RESULTS_PKL, "rb") as f:
        ALL_RESULTS = pickle.load(f)
    print(f"Loaded existing ALL_RESULTS with {len(ALL_RESULTS)} entries")
else:
    print("Starting fresh ALL_RESULTS")

print("Output path:", OUT)

Starting fresh ALL_RESULTS
Output path: /content/drive/MyDrive/FinalProjectDL/final_run


## 3. Dataset Check

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def list_images(folder):
    folder = Path(folder)
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)

def read_class_map(prompts_file):
    """Parse prompts_and_classes.txt into {subject_name: class_noun}."""
    class_map = {}
    in_classes = False
    with open(prompts_file, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("subject_name,class"):
                in_classes = True
                continue
            if in_classes and "," in line:
                parts = line.split(",", 1)
                if len(parts) == 2:
                    class_map[parts[0].strip()] = parts[1].strip()
            elif in_classes and not line:
                break
    return class_map

# Verify dataset
assert DATASET_ROOT.exists(), f"Dataset not found at {DATASET_ROOT}"
CLASS_MAP = read_class_map(DATASET_ROOT / "prompts_and_classes.txt")

for subj in CFG["subjects"]:
    name = subj["name"]
    imgs = list_images(DATASET_ROOT / name)
    expected_class = CLASS_MAP.get(name, "???")
    print(f"  {name:25s} -> {len(imgs)} images, "
          f"class: '{subj['class_noun']}' (file says '{expected_class}')")
print("\n✓ Dataset OK")


  dog                       -> 5 images, class: 'dog' (file says 'dog')
  bear_plushie              -> 5 images, class: 'stuffed animal' (file says 'stuffed animal')
  colorful_sneaker          -> 5 images, class: 'sneaker' (file says 'sneaker')
  teapot                    -> 5 images, class: 'teapot' (file says 'teapot')

✓ Dataset OK


## 4. Helper Functions

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def safe_name(text):
    return text.replace(" ", "_").replace("/", "_")

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def instance_prompt(class_noun, identifier="sks"):
    return f"a photo of a {identifier} {class_noun}"

def class_prompt(class_noun):
    return f"a photo of a {class_noun}"

def save_all_results():
    with open(RESULTS_PKL, "wb") as f:
        pickle.dump(ALL_RESULTS, f)


In [ ]:
def get_eval_prompts(class_noun, is_living, identifier="sks"):
    ending = "wearing a santa hat" if is_living else "in a bucket"
    return [
        f"a photo of a {identifier} {class_noun} in the snow",
        f"a photo of a {identifier} {class_noun} on the beach",
        f"a photo of a {identifier} {class_noun} in Times Square",
        f"a photo of a {identifier} {class_noun} on top of a mountain",
        f"a photo of a {identifier} {class_noun} with a city in the background",
        f"a watercolor painting of a {identifier} {class_noun}",
        f"a painting of a {identifier} {class_noun} in the style of Van Gogh",
        f"a photo of a {identifier} {class_noun} {ending}",
    ]

def get_font(size=18):
    try:
        return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size)
    except Exception:
        return ImageFont.load_default()

def make_labeled_grid(image_rows, row_labels, save_path, title=None, cell_size=256):
    """Build a labeled grid image: len(image_rows) rows x len(image_rows[0]) cols."""
    n_rows = len(image_rows)
    n_cols = max(len(r) for r in image_rows)
    label_w = 320
    grid_w = label_w + n_cols * cell_size
    grid_h = n_rows * cell_size
    grid = Image.new("RGB", (grid_w, grid_h), (255, 255, 255))
    draw = ImageDraw.Draw(grid)
    font = get_font(11)

    for r, (imgs, label) in enumerate(zip(image_rows, row_labels)):
        y = r * cell_size
        # Wrap label text
        draw.text((5, y + cell_size // 3), label[:60], fill=(0, 0, 0), font=font)
        for c, img in enumerate(imgs):
            if isinstance(img, (str, Path)):
                img = Image.open(img).convert("RGB")
            img = img.resize((cell_size, cell_size), Image.BICUBIC)
            grid.paste(img, (label_w + c * cell_size, y))

    if title:
        # Simple title at top - just save with title in filename
        pass
    grid.save(save_path)
    return save_path

def stitch_existing_grids(grid_paths, labels, save_path):
    """Stitch multiple grid PNGs side by side with labels."""
    imgs = [Image.open(p) for p in grid_paths if Path(p).exists()]
    if not imgs:
        print(f"  No grids found for {save_path}")
        return
    widths = [img.width for img in imgs]
    max_h = max(img.height for img in imgs)
    label_h = 40
    total_w = sum(widths)
    canvas = Image.new("RGB", (total_w, max_h + label_h), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    font = get_font(16)
    x = 0
    for img, label in zip(imgs, labels):
        draw.text((x + 10, 5), label, fill=(0, 0, 0), font=font)
        canvas.paste(img, (x, label_h))
        x += img.width
    canvas.save(save_path)
    print(f"  Saved {save_path}")

print("✓ Utilities defined")


✓ Utilities defined


In [ ]:
class DreamBoothDataset(Dataset):
    """
    DreamBooth training dataset.
    instance_sources: list of (folder_path, prompt_string) pairs.
    Supports both single-subject and multi-subject mixing natively.
    """
    def __init__(self, instance_sources, tokenizer, resolution,
                 class_dir=None, class_prompt_text=None,
                 with_prior=True, n_images=None):
        self.tokenizer = tokenizer
        self.with_prior = with_prior
        self.class_prompt_text = class_prompt_text

        # Collect all instance images with their prompts
        self.instance_items = []
        for folder, prompt_text in instance_sources:
            imgs = list_images(folder)
            if n_images is not None:
                imgs = imgs[:n_images]
            for img_path in imgs:
                self.instance_items.append((img_path, prompt_text))

        if len(self.instance_items) == 0:
            raise ValueError("No instance images found")

        self.class_images = []
        if with_prior:
            self.class_images = list_images(class_dir)
            if len(self.class_images) == 0:
                raise ValueError(f"No class images in {class_dir}")

        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        if self.with_prior:
            return max(len(self.instance_items), len(self.class_images))
        return len(self.instance_items)

    def tokenize(self, prompt):
        return self.tokenizer(
            prompt, padding="max_length", truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors="pt",
        ).input_ids[0]

    def read_image(self, path):
        return self.transform(Image.open(path).convert("RGB"))

    def __getitem__(self, idx):
        inst_path, inst_prompt = self.instance_items[idx % len(self.instance_items)]
        item = {
            "instance_pixel_values": self.read_image(inst_path),
            "instance_input_ids": self.tokenize(inst_prompt),
        }
        if self.with_prior:
            cls_path = self.class_images[idx % len(self.class_images)]
            item["class_pixel_values"] = self.read_image(cls_path)
            item["class_input_ids"] = self.tokenize(self.class_prompt_text)
        return item

print("✓ DreamBoothDataset defined (handles single + multi-source natively)")


✓ DreamBoothDataset defined (handles single + multi-source natively)


In [ ]:
def load_training_components(train_text_encoder=False):
    """Load all model components for training. Returns on CUDA, float32."""
    model_id = CFG["pretrained_model"]
    device = get_device()

    tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)
    vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device)
    unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to(device)
    noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

    vae.requires_grad_(False)
    vae.eval()

    if train_text_encoder:
        text_encoder.requires_grad_(True)
    else:
        text_encoder.requires_grad_(False)
        text_encoder.eval()

    unet.requires_grad_(True)
    return tokenizer, text_encoder, vae, unet, noise_scheduler


def build_inference_pipe(tokenizer, text_encoder, vae, unet):
    """Build fp16 inference pipeline from trained components (no re-download)."""
    pipe = StableDiffusionPipeline(
        vae=vae,
        text_encoder=text_encoder,
        tokenizer=tokenizer,
        unet=unet,
        scheduler=DPMSolverMultistepScheduler.from_pretrained(
            CFG["pretrained_model"], subfolder="scheduler"
        ),
        safety_checker=None,
        feature_extractor=None,
        requires_safety_checker=False,
    )
    pipe.unet.half()
    pipe.text_encoder.half()
    pipe.vae.half()
    pipe = pipe.to(get_device())
    pipe.set_progress_bar_config(disable=True)
    return pipe

print("✓ Training component loaders defined")


✓ Training component loaders defined


In [ ]:
def train_dreambooth(run_dir, instance_sources, class_noun, with_prior=True,
                     train_text_encoder=False, n_images=None, max_steps=None):
    """
    Train DreamBooth. Returns inference pipeline directly (no CPU state dict copy).
    Concatenates instance + class into one batch for a single forward pass.
    Tracks instance_loss and prior_loss separately.
    """
    seed_everything(CFG["seed"])
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    max_steps = max_steps or CFG["max_train_steps"]
    lr = CFG["lr_te"] if train_text_encoder else CFG["lr_unet"]
    device = get_device()

    tokenizer, text_encoder, vae, unet, noise_scheduler = \
        load_training_components(train_text_encoder)

    class_dir = CLASS_ROOT / safe_name(class_noun)
    dataset = DreamBoothDataset(
        instance_sources=instance_sources,
        tokenizer=tokenizer,
        resolution=CFG["resolution"],
        class_dir=class_dir,
        class_prompt_text=class_prompt(class_noun),
        with_prior=with_prior,
        n_images=n_images,
    )
    loader = DataLoader(dataset, batch_size=CFG["train_batch_size"],
                        shuffle=True, num_workers=0)

    params = list(unet.parameters())
    if train_text_encoder:
        params += list(text_encoder.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-2)

    loss_rows = []
    global_step = 0
    unet.train()
    if train_text_encoder:
        text_encoder.train()

    print(f"\n{'='*60}")
    print(f"Training: {run_dir.name}")
    print(f"  Prior: {with_prior}, Train TE: {train_text_encoder}, "
          f"n_images: {n_images or 'all'}, steps: {max_steps}, lr: {lr}")
    print(f"{'='*60}")

    while global_step < max_steps:
        for batch in loader:
            if global_step >= max_steps:
                break

            inst_pixels = batch["instance_pixel_values"].to(device, dtype=torch.float32)
            inst_ids = batch["instance_input_ids"].to(device)

            # Concatenate instance + class into one batch for single forward pass
            if with_prior:
                cls_pixels = batch["class_pixel_values"].to(device, dtype=torch.float32)
                cls_ids = batch["class_input_ids"].to(device)
                pixel_values = torch.cat([inst_pixels, cls_pixels], dim=0)
                input_ids = torch.cat([inst_ids, cls_ids], dim=0)
            else:
                pixel_values = inst_pixels
                input_ids = inst_ids

            # Encode -> noise -> denoise (single forward pass)
            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            encoder_hidden_states = text_encoder(input_ids)[0]
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            # Chunk predictions and compute separate losses
            if with_prior:
                pred_inst, pred_cls = model_pred.chunk(2)
                noise_inst, noise_cls = noise.chunk(2)
                instance_loss = F.mse_loss(pred_inst.float(), noise_inst.float(), reduction="mean")
                prior_loss = F.mse_loss(pred_cls.float(), noise_cls.float(), reduction="mean")
                loss = instance_loss + CFG["lambda_prior"] * prior_loss
                prior_value = float(prior_loss.detach().cpu())
            else:
                instance_loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
                loss = instance_loss
                prior_value = np.nan

            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, CFG["max_grad_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            global_step += 1
            if global_step % 25 == 0 or global_step == 1 or global_step == max_steps:
                row = {
                    "step": global_step,
                    "loss": float(loss.detach().cpu()),
                    "instance_loss": float(instance_loss.detach().cpu()),
                    "prior_loss": prior_value,
                }
                loss_rows.append(row)
                if global_step % 100 == 0 or global_step == 1:
                    print(f"  step {global_step}: loss={row['loss']:.4f} "
                          f"inst={row['instance_loss']:.4f} prior={prior_value:.4f}")

    # Save loss history
    loss_df = pd.DataFrame(loss_rows)
    loss_df.to_csv(run_dir / "loss_history.csv", index=False)

    # Plot loss curve with separate instance/prior lines
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(loss_df["step"], loss_df["loss"], label="total", linewidth=1.5)
    ax.plot(loss_df["step"], loss_df["instance_loss"], label="instance",
            linewidth=1, alpha=0.8)
    if with_prior:
        ax.plot(loss_df["step"], loss_df["prior_loss"], label="prior",
                linewidth=1, alpha=0.8)
    ax.set_xlabel("Step")
    ax.set_ylabel("MSE Loss")
    ax.set_title(run_dir.name)
    ax.legend()
    plt.tight_layout()
    fig.savefig(run_dir / "loss_curve.png", dpi=180, bbox_inches="tight")
    plt.close(fig)

    # Build inference pipeline directly from trained weights (no CPU copy)
    pipe = build_inference_pipe(tokenizer, text_encoder, vae, unet)
    return pipe

print("✓ train_dreambooth() defined — single forward pass, separate loss tracking")


✓ train_dreambooth() defined — single forward pass, separate loss tracking


In [ ]:
@torch.no_grad()
def generate_images(pipe, prompts, out_folder, images_per_prompt=4, prefix="img"):
    """Generate images for a list of prompts, save to disk, return paths grouped by prompt."""
    out_folder = Path(out_folder)
    out_folder.mkdir(parents=True, exist_ok=True)
    all_paths = []
    generator = torch.Generator(device=get_device()).manual_seed(CFG["seed"])

    for p_idx, prompt in enumerate(prompts):
        prompt_paths = []
        for i_idx in range(images_per_prompt):
            img = pipe(
                prompt,
                num_inference_steps=CFG["sample_steps"],
                guidance_scale=CFG["guidance_scale"],
                generator=generator,
            ).images[0]
            path = out_folder / f"{prefix}_p{p_idx:02d}_i{i_idx}.png"
            img.save(path)
            prompt_paths.append(str(path))
        all_paths.append(prompt_paths)

    return all_paths


def generate_run_outputs(pipe, run_dir, class_noun, is_living, identifiers=None):
    """Generate eval grid, drift images. Returns dict of paths."""
    identifiers = identifiers or [CFG["identifier"]]
    run_dir = Path(run_dir)

    all_generations = []
    all_prompts = []

    for ident in identifiers:
        prompts = get_eval_prompts(class_noun, is_living, identifier=ident)
        gen_dir = run_dir / "generated"
        generations = generate_images(
            pipe, prompts, gen_dir,
            images_per_prompt=CFG["images_per_prompt"],
            prefix=ident,
        )
        all_generations.extend(generations)
        all_prompts.extend(prompts)

        # Build labeled grid
        image_rows = []
        for prompt, paths in zip(prompts, generations):
            image_rows.append([Image.open(p) for p in paths])
        make_labeled_grid(
            image_rows, prompts,
            run_dir / "grid.png",
            title=f"{run_dir.parent.name}/{run_dir.name}",
        )

    # Drift images: class prompt with no identifier
    drift_prompt = class_prompt(class_noun)
    drift_dir = run_dir / "drift"
    drift_paths = generate_images(
        pipe, [drift_prompt], drift_dir,
        images_per_prompt=4, prefix="drift",
    )
    # Save individual drift PNGs to run_dir too for easy access
    for i, p in enumerate(drift_paths[0]):
        shutil.copy2(p, run_dir / f"drift_{i}.png")

    return {
        "generations": all_generations,
        "prompts": all_prompts,
        "drift_images": [str(p) for p in drift_paths[0]],
    }

print("✓ Inference + output generation defined")


✓ Inference + output generation defined


In [ ]:
def append_metrics_row(row):
    """Append one metrics row to CSV immediately (crash-resilient)."""
    row = dict(row)
    row["timestamp"] = datetime.now().isoformat(timespec="seconds")
    df = pd.DataFrame([row])
    header = not METRICS_CSV.exists()
    df.to_csv(METRICS_CSV, mode="a", header=header, index=False)
    print(f"  -> appended metrics: {row['run_key']}")


def cosine_mean(a, b):
    a = F.normalize(a, dim=-1)
    b = F.normalize(b, dim=-1)
    return float((a @ b.T).mean().detach().cpu())


def load_metric_models():
    device = get_device()
    dino = torch.hub.load(CFG["dino_repo"], CFG["dino_model"]).to(device).eval()
    clip_model = CLIPModel.from_pretrained(CFG["clip_model_name"]).to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained(CFG["clip_model_name"])
    lpips_model = lpips.LPIPS(net="alex").to(device).eval()
    return dino, clip_model, clip_processor, lpips_model


def dino_features(dino, paths, batch_size=16):
    """Extract DINO features with proper ImageNet normalization."""
    device = get_device()
    tfm = transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    feats = []
    for i in range(0, len(paths), batch_size):
        imgs = [tfm(Image.open(p).convert("RGB")) for p in paths[i:i+batch_size]]
        x = torch.stack(imgs).to(device)
        with torch.no_grad():
            feats.append(dino(x).detach().cpu())
    return torch.cat(feats, dim=0)


def clip_image_features(clip_model, clip_processor, paths, batch_size=16):
    device = get_device()
    feats = []
    for i in range(0, len(paths), batch_size):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i+batch_size]]
        inputs = clip_processor(images=imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            f = clip_model.get_image_features(**inputs)
        feats.append(f.detach().cpu())
    return torch.cat(feats, dim=0)


def clip_text_image_score(clip_model, clip_processor, image_paths, prompts):
    device = get_device()
    scores = []
    for img_path, prompt in zip(image_paths, prompts):
        img = Image.open(img_path).convert("RGB")
        inputs = clip_processor(text=[prompt], images=[img],
                                return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            img_f = clip_model.get_image_features(pixel_values=inputs["pixel_values"])
            txt_f = clip_model.get_text_features(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
            )
        scores.append(cosine_mean(img_f.cpu(), txt_f.cpu()))
    return float(np.mean(scores))


def lpips_diversity(lpips_model, image_paths_by_prompt):
    device = get_device()
    tfm = transforms.Compose([
        transforms.Resize((256, 256), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])
    per_prompt = []
    for paths in image_paths_by_prompt:
        imgs = [tfm(Image.open(p).convert("RGB")).to(device) for p in paths]
        dists = []
        for i in range(len(imgs)):
            for j in range(i + 1, len(imgs)):
                with torch.no_grad():
                    d = lpips_model(imgs[i].unsqueeze(0), imgs[j].unsqueeze(0))
                dists.append(float(d.detach().cpu()))
        if dists:
            per_prompt.append(np.mean(dists))
    return float(np.mean(per_prompt)) if per_prompt else np.nan


def score_run(run_key, result):
    """Score a completed run. Loads metric models fresh, frees them after."""
    print(f"  Scoring {run_key}...")
    free_memory()
    dino, clip_model, clip_processor, lpips_model = load_metric_models()

    ref_paths = result["reference_images"]
    image_paths_by_prompt = result["generations"]
    gen_paths = [p for row in image_paths_by_prompt for p in row]
    gen_prompts = []
    for prompt, row in zip(result["prompts"], image_paths_by_prompt):
        gen_prompts.extend([prompt] * len(row))
    drift_paths = result["drift_images"]

    ref_dino = dino_features(dino, ref_paths)
    gen_dino = dino_features(dino, gen_paths)
    drift_dino = dino_features(dino, drift_paths)

    ref_clip = clip_image_features(clip_model, clip_processor, ref_paths)
    gen_clip = clip_image_features(clip_model, clip_processor, gen_paths)

    row = {
        "run_key": run_key,
        "subject": result["subject"],
        "run_name": result["run_name"],
        "class_noun": result["class_noun"],
        "n_images": result["n_images"],
        "with_prior": result["with_prior"],
        "train_text_encoder": result["train_text_encoder"],
        "DINO": round(cosine_mean(gen_dino, ref_dino), 4),
        "CLIP_I": round(cosine_mean(gen_clip, ref_clip), 4),
        "CLIP_T": round(clip_text_image_score(clip_model, clip_processor,
                                               gen_paths, gen_prompts), 4),
        "PRES": round(cosine_mean(drift_dino, ref_dino), 4),
        "DIV": round(lpips_diversity(lpips_model, image_paths_by_prompt), 4),
    }
    append_metrics_row(row)

    del dino, clip_model, clip_processor, lpips_model
    free_memory()
    return row

print("✓ Metrics defined (DINO w/ ImageNet norm, CLIP-I, CLIP-T, PRES, DIV)")


✓ Metrics defined (DINO w/ ImageNet norm, CLIP-I, CLIP-T, PRES, DIV)


In [ ]:
def write_metadata(run_dir, metadata):
    lines = ["DreamBooth final run metadata", "=" * 32]
    for k, v in metadata.items():
        lines.append(f"{k}: {v}")
    lines.append("\nFull config snapshot:")
    lines.append(json.dumps(CFG, indent=2, default=str))
    with open(Path(run_dir) / "metadata.txt", "w") as f:
        f.write("\n".join(lines))


def run_experiment(subject_name, run_name, class_noun=None, is_living=None,
                   n_images=None, with_prior=True, train_text_encoder=False,
                   instance_sources=None, eval_identifiers=None,
                   score=True, return_pipe=False):
    """
    Full experiment orchestrator: train -> infer -> score -> save.
    Returns result dict (and optionally the pipe for reuse).
    Scores immediately after each run (crash-resilient).
    Unloads SD pipe before loading metric models (T4-safe).
    """
    class_noun = class_noun or CLASS_MAP.get(subject_name, "object")
    if is_living is None:
        is_living = class_noun in ["dog", "cat"]
    run_key = f"{subject_name}_{run_name}"
    run_dir = OUT / subject_name / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # Skip if already completed
    if run_key in ALL_RESULTS:
        print(f"⏭ Skipping {run_key} (already done)")
        return ALL_RESULTS[run_key]

    if instance_sources is None:
        instance_sources = [
            (DATASET_ROOT / subject_name,
             instance_prompt(class_noun, CFG["identifier"]))
        ]

    # Collect reference images
    reference_images = []
    for folder, _ in instance_sources:
        imgs = list_images(folder)
        if n_images is not None:
            imgs = imgs[:n_images]
        reference_images.extend(imgs)

    metadata = {
        "run_key": run_key,
        "subject": subject_name,
        "run_name": run_name,
        "class_noun": class_noun,
        "n_images": n_images if n_images is not None else "all_available",
        "reference_image_count": len(reference_images),
        "with_prior": with_prior,
        "train_text_encoder": train_text_encoder,
        "learning_rate": CFG["lr_te"] if train_text_encoder else CFG["lr_unet"],
        "instance_sources": [(str(a), b) for a, b in instance_sources],
        "started": datetime.now().isoformat(timespec="seconds"),
    }
    write_metadata(run_dir, metadata)

    # Train — returns pipe directly
    pipe = train_dreambooth(
        run_dir=run_dir,
        instance_sources=instance_sources,
        class_noun=class_noun,
        with_prior=with_prior,
        train_text_encoder=train_text_encoder,
        n_images=n_images,
        max_steps=CFG["max_train_steps"],
    )

    # Inference
    output_info = generate_run_outputs(
        pipe=pipe,
        run_dir=run_dir,
        class_noun=class_noun,
        is_living=is_living,
        identifiers=eval_identifiers or [CFG["identifier"]],
    )

    result = {
        "run_key": run_key,
        "subject": subject_name,
        "run_name": run_name,
        "class_noun": class_noun,
        "is_living": is_living,
        "n_images": n_images if n_images is not None else len(reference_images),
        "with_prior": with_prior,
        "train_text_encoder": train_text_encoder,
        "reference_images": [str(p) for p in reference_images],
        "run_dir": str(run_dir),
        **output_info,
    }
    ALL_RESULTS[run_key] = result
    save_all_results()

    # Score — unload pipe first to free VRAM for metric models
    metric_row = None
    if score and not return_pipe:
        del pipe
        free_memory()
        metric_row = score_run(run_key, result)
        ALL_RESULTS[run_key]["metrics"] = metric_row
        save_all_results()
        return result

    if score:
        metric_row = score_run(run_key, result)
        ALL_RESULTS[run_key]["metrics"] = metric_row
        save_all_results()

    if return_pipe:
        return result, pipe

    del pipe
    free_memory()
    return result

print("✓ run_experiment() orchestrator defined")


✓ run_experiment() orchestrator defined


## 5. Vanilla SD Baseline Generation

In [ ]:
def generate_vanilla_baselines():
    seed_everything(CFG["seed"])
    device = get_device()
    pipe = StableDiffusionPipeline.from_pretrained(
        CFG["pretrained_model"],
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    generator = torch.Generator(device=device).manual_seed(CFG["seed"])
    for subj in CFG["subjects"]:
        name, cn = subj["name"], subj["class_noun"]
        out_dir = VANILLA_ROOT / name
        out_dir.mkdir(parents=True, exist_ok=True)
        if (out_dir / "img_3.png").exists():
            print(f"  {name}: already exists, skipping")
            continue
        prompt = class_prompt(cn)
        for i in range(4):
            img = pipe(prompt, num_inference_steps=CFG["sample_steps"],
                       guidance_scale=CFG["guidance_scale"],
                       generator=generator).images[0]
            img.save(out_dir / f"img_{i}.png")
        print(f"  {name}: saved 4 vanilla images ('{prompt}')")

    del pipe
    free_memory()
    print("✓ Vanilla baselines ready")

generate_vanilla_baselines()


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  dog: saved 4 vanilla images ('a photo of a dog')
  bear_plushie: saved 4 vanilla images ('a photo of a stuffed animal')
  colorful_sneaker: saved 4 vanilla images ('a photo of a sneaker')
  teapot: saved 4 vanilla images ('a photo of a teapot')
✓ Vanilla baselines ready


## 6. Class Image Generation

In [ ]:
def generate_class_images_for_class(class_noun):
    class_dir = CLASS_ROOT / safe_name(class_noun)
    class_dir.mkdir(parents=True, exist_ok=True)
    existing = list_images(class_dir)
    needed = CFG["num_class_images"] - len(existing)
    if needed <= 0:
        print(f"  '{class_noun}': {len(existing)} images exist, skipping")
        return

    print(f"  Generating {needed} class images for '{class_noun}'...")
    device = get_device()
    pipe = StableDiffusionPipeline.from_pretrained(
        CFG["pretrained_model"], torch_dtype=torch.float16,
        safety_checker=None, requires_safety_checker=False,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    prompt = class_prompt(class_noun)
    count = len(existing)
    generator = torch.Generator(device=device).manual_seed(CFG["seed"])
    batch = CFG["class_gen_batch"]

    while count < CFG["num_class_images"]:
        n = min(batch, CFG["num_class_images"] - count)
        imgs = pipe([prompt] * n, num_inference_steps=CFG["class_gen_steps"],
                    guidance_scale=CFG["guidance_scale"],
                    generator=generator).images
        for img in imgs:
            img.save(class_dir / f"class_{count:04d}.png")
            count += 1

    del pipe
    free_memory()
    print(f"    -> {count} total images in {class_dir}")

# Generate for all unique class nouns
unique_classes = list(set(s["class_noun"] for s in CFG["subjects"]))
print("Class nouns:", unique_classes)
for cn in unique_classes:
    generate_class_images_for_class(cn)
print("✓ Class images ready")


Class nouns: ['dog', 'sneaker', 'stuffed animal', 'teapot']
  Generating 200 class images for 'dog'...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

    -> 200 total images in /content/drive/MyDrive/FinalProjectDL/final_run/class_images_final/dog
  Generating 200 class images for 'sneaker'...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

    -> 200 total images in /content/drive/MyDrive/FinalProjectDL/final_run/class_images_final/stuffed_animal
  Generating 200 class images for 'teapot'...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

    -> 200 total images in /content/drive/MyDrive/FinalProjectDL/final_run/class_images_final/teapot
✓ Class images ready


## 7. Main Runs Loop: 4 subjects × main + no_prior

In [ ]:
# ── FIX 1: CLIP output compatibility ──
def _to_tensor(output):
    """Handle both raw tensors and output objects from CLIP."""
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, "pooler_output"):
        return output.pooler_output
    return output[1]

def clip_image_features(clip_model, clip_processor, paths, batch_size=16):
    device = get_device()
    feats = []
    for i in range(0, len(paths), batch_size):
        imgs = [Image.open(p).convert("RGB") for p in paths[i:i+batch_size]]
        inputs = clip_processor(images=imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            f = _to_tensor(clip_model.get_image_features(pixel_values=inputs["pixel_values"]))
        feats.append(f.detach().cpu())
    return torch.cat(feats, dim=0)

def clip_text_image_score(clip_model, clip_processor, image_paths, prompts):
    device = get_device()
    scores = []
    for img_path, prompt in zip(image_paths, prompts):
        img = Image.open(img_path).convert("RGB")
        inputs = clip_processor(text=[prompt], images=[img],
                                return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            img_f = _to_tensor(clip_model.get_image_features(pixel_values=inputs["pixel_values"]))
            txt_f = _to_tensor(clip_model.get_text_features(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
            ))
        scores.append(cosine_mean(img_f.cpu(), txt_f.cpu()))
    return float(np.mean(scores))


# ── FIX 3: Score the already-completed dog_main ──
if "dog_main" in ALL_RESULTS and "metrics" not in ALL_RESULTS["dog_main"]:
    metric_row = score_run("dog_main", ALL_RESULTS["dog_main"])
    ALL_RESULTS["dog_main"]["metrics"] = metric_row
    save_all_results()
    print("✓ dog_main scored and saved")
else:
    print("dog_main either not found or already scored")

print("✓ Fixes applied. Re-run the main loop cell now.")

dog_main either not found or already scored
✓ Fixes applied. Re-run the main loop cell now.


In [ ]:
for subj in CFG["subjects"]:
    name = subj["name"]
    cn = subj["class_noun"]
    living = subj["is_living"]

    print("\n" + "=" * 80)
    print("MAIN:", name)
    run_experiment(
        subject_name=name, run_name="main",
        class_noun=cn, is_living=living,
        n_images=None, with_prior=True, train_text_encoder=False,
    )

    print("\n" + "=" * 80)
    print("NO PRIOR:", name)
    run_experiment(
        subject_name=name, run_name="no_prior",
        class_noun=cn, is_living=living,
        n_images=None, with_prior=False, train_text_encoder=False,
    )

print("\n✓ All 8 main runs complete.")



MAIN: dog


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: main
  Prior: True, Train TE: False, n_images: all, steps: 800, lr: 3e-06
  step 1: loss=1.3587 inst=0.6608 prior=0.6979
  step 100: loss=0.1007 inst=0.0952 prior=0.0054
  step 200: loss=0.4436 inst=0.1907 prior=0.2528
  step 300: loss=0.0975 inst=0.0272 prior=0.0703
  step 400: loss=0.3788 inst=0.1150 prior=0.2638
  step 500: loss=0.5573 inst=0.0536 prior=0.5036
  step 600: loss=0.1618 inst=0.1349 prior=0.0269
  step 700: loss=0.3931 inst=0.1167 prior=0.2764
  step 800: loss=0.0417 inst=0.0267 prior=0.0149
  Scoring dog_main...
Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_deitsmall16_pretrain.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 228MB/s]


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: no_prior
  Prior: False, Train TE: False, n_images: all, steps: 800, lr: 3e-06
  step 1: loss=0.5101 inst=0.5101 prior=nan
  step 100: loss=0.0973 inst=0.0973 prior=nan
  step 200: loss=0.1793 inst=0.1793 prior=nan
  step 300: loss=0.0201 inst=0.0201 prior=nan
  step 400: loss=0.0918 inst=0.0918 prior=nan
  step 500: loss=0.0616 inst=0.0616 prior=nan
  step 600: loss=0.0923 inst=0.0923 prior=nan
  step 700: loss=0.0936 inst=0.0936 prior=nan
  step 800: loss=0.0149 inst=0.0149 prior=nan
  Scoring teapot_no_prior...


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
  -> appended metrics: teapot_no_prior

✓ All 8 main runs complete.


## 8. Dog Ablations: n=3, n=4, text encoder unfrozen

In [ ]:
dog_cfg = next(s for s in CFG["subjects"] if s["name"] == CFG["headline_subject"])

run_experiment(
    subject_name="dog", run_name="n3",
    class_noun=dog_cfg["class_noun"], is_living=dog_cfg["is_living"],
    n_images=3, with_prior=True, train_text_encoder=False,
)

run_experiment(
    subject_name="dog", run_name="n4",
    class_noun=dog_cfg["class_noun"], is_living=dog_cfg["is_living"],
    n_images=4, with_prior=True, train_text_encoder=False,
)

run_experiment(
    subject_name="dog", run_name="te_unfrozen",
    class_noun=dog_cfg["class_noun"], is_living=dog_cfg["is_living"],
    n_images=None, with_prior=True, train_text_encoder=True,
)

print("\n✓ All 3 dog ablations complete.")


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: n3
  Prior: True, Train TE: False, n_images: 3, steps: 800, lr: 3e-06
  step 1: loss=1.4584 inst=0.7605 prior=0.6979
  step 100: loss=0.1030 inst=0.0976 prior=0.0055
  step 200: loss=0.4581 inst=0.2052 prior=0.2529
  step 300: loss=0.0962 inst=0.0260 prior=0.0702
  step 400: loss=0.3917 inst=0.1278 prior=0.2639
  step 500: loss=0.5559 inst=0.0510 prior=0.5049
  step 600: loss=0.1598 inst=0.1328 prior=0.0270
  step 700: loss=0.3953 inst=0.1191 prior=0.2762
  step 800: loss=0.0329 inst=0.0183 prior=0.0146
  Scoring dog_n3...


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
  -> appended metrics: dog_n3


...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: te_unfrozen
  Prior: True, Train TE: True, n_images: all, steps: 800, lr: 2e-06
  step 1: loss=1.3587 inst=0.6608 prior=0.6979
  step 100: loss=0.1015 inst=0.0960 prior=0.0055
  step 200: loss=0.4472 inst=0.1949 prior=0.2522
  step 300: loss=0.0977 inst=0.0275 prior=0.0702
  step 400: loss=0.3845 inst=0.1211 prior=0.2634
  step 500: loss=0.5593 inst=0.0566 prior=0.5027
  step 600: loss=0.1723 inst=0.1454 prior=0.0269
  step 700: loss=0.4054 inst=0.1293 prior=0.2761
  step 800: loss=0.0448 inst=0.0300 prior=0.0148
  Scoring dog_te_unfrozen...


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
  -> appended metrics: dog_te_unfrozen

✓ All 3 dog ablations complete.


## 9. Per-Subject Prior Preservation Figures

In [ ]:
def make_prior_vs_no_prior(subject):
    subj_dir = OUT / subject
    vanilla = sorted((VANILLA_ROOT / subject).glob("img_*.png"))[:4]
    no_prior = [Path(p) for p in ALL_RESULTS[f"{subject}_no_prior"]["drift_images"]]
    main = [Path(p) for p in ALL_RESULTS[f"{subject}_main"]["drift_images"]]

    cn = next(s["class_noun"] for s in CFG["subjects"] if s["name"] == subject)
    drift_prompt = f"a photo of a {cn}"

    make_labeled_grid(
        [vanilla, no_prior, main],
        [
            f"Vanilla SD: '{drift_prompt}'",
            f"No prior pres. drift",
            f"With prior pres. drift",
        ],
        subj_dir / "prior_vs_no_prior.png",
    )
    print(f"  Saved {subj_dir / 'prior_vs_no_prior.png'}")

for subj in CFG["subjects"]:
    make_prior_vs_no_prior(subj["name"])

print("✓ All prior_vs_no_prior figures saved.")


  Saved /content/drive/MyDrive/FinalProjectDL/final_run/dog/prior_vs_no_prior.png
  Saved /content/drive/MyDrive/FinalProjectDL/final_run/bear_plushie/prior_vs_no_prior.png
  Saved /content/drive/MyDrive/FinalProjectDL/final_run/colorful_sneaker/prior_vs_no_prior.png
  Saved /content/drive/MyDrive/FinalProjectDL/final_run/teapot/prior_vs_no_prior.png
✓ All prior_vs_no_prior figures saved.


## 10. Dog Visual Comparison Figures

In [ ]:
dog_dir = OUT / "dog"

n_sweep_path = dog_dir / "n_sweep.png"
stitch_existing_grids(
    [dog_dir / "n3" / "grid.png",
     dog_dir / "n4" / "grid.png",
     dog_dir / "main" / "grid.png"],
    ["n=3", "n=4", "n=5/all"],
    n_sweep_path,
)

te_compare_path = dog_dir / "te_compare.png"
stitch_existing_grids(
    [dog_dir / "main" / "grid.png",
     dog_dir / "te_unfrozen" / "grid.png"],
    ["TE Frozen (main)", "TE Trained"],
    te_compare_path,
)

print("✓ Dog ablation figures saved.")


  Saved /content/drive/MyDrive/FinalProjectDL/final_run/dog/n_sweep.png
  Saved /content/drive/MyDrive/FinalProjectDL/final_run/dog/te_compare.png
✓ Dog ablation figures saved.


## 11. Global Metrics Table

In [ ]:
def make_metrics_table():
    df = pd.read_csv(METRICS_CSV)
    metric_cols = ["DINO", "CLIP_I", "CLIP_T", "PRES", "DIV"]
    show_cols = ["subject", "run_name", "n_images", "with_prior",
                 "train_text_encoder"] + metric_cols
    table_df = df[[c for c in show_cols if c in df.columns]].copy()
    for c in metric_cols:
        if c in table_df.columns:
            table_df[c] = table_df[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "")

    print(table_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(14, max(3, len(table_df) * 0.45 + 1)))
    ax.axis("off")
    table = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        cellLoc="center", loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.2, 1.5)
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#4472C4")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#D9E2F3")
    plt.title("DreamBooth Reproduction — All Metrics", fontsize=14, pad=20)
    out = OUT / "metrics_table.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n✓ Metrics table saved to {out}")

make_metrics_table()


         subject    run_name  n_images  with_prior  train_text_encoder   DINO CLIP_I CLIP_T   PRES    DIV
             dog        main         5        True               False 0.6635 0.8046 0.3100 0.2360 0.6624
             dog    no_prior         5       False               False 0.7873 0.8409 0.2962 0.8739 0.6037
    bear_plushie        main         5        True               False 0.6412 0.7158 0.3455 0.3743 0.6285
    bear_plushie    no_prior         5       False               False 0.7488 0.7775 0.3150 0.7984 0.5123
colorful_sneaker        main         5        True               False 0.6827 0.7512 0.3218 0.6070 0.6480
colorful_sneaker    no_prior         5       False               False 0.7886 0.8146 0.2861 0.8089 0.5193
          teapot        main         5        True               False 0.4440 0.7837 0.3645 0.4496 0.7180
          teapot    no_prior         5       False               False 0.4795 0.7975 0.3419 0.5582 0.6744
             dog          n3         3        

## 12. Final Summary

In [ ]:
print("Final output root:", OUT)

print("\nGlobal files:")
for p in [METRICS_CSV, OUT / "metrics_table.png", RESULTS_PKL]:
    print(" ", p)

print("\nPer-subject figures:")
for subj in CFG["subjects"]:
    name = subj["name"]
    print(f"\n  {name}")
    print(f"    main grid:       {OUT / name / 'main' / 'grid.png'}")
    print(f"    no_prior grid:   {OUT / name / 'no_prior' / 'grid.png'}")
    print(f"    prior_vs_no:     {OUT / name / 'prior_vs_no_prior.png'}")

print(f"\n  dog ablations:")
print(f"    n_sweep:         {OUT / 'dog' / 'n_sweep.png'}")
print(f"    te_compare:      {OUT / 'dog' / 'te_compare.png'}")

print(f"\nTotal runs completed: {len(ALL_RESULTS)}")
print("\n✓ All done! Copy figures from final_run/ for your paper.")


Final output root: /content/drive/MyDrive/FinalProjectDL/final_run

Global files:
  /content/drive/MyDrive/FinalProjectDL/final_run/all_metrics.csv
  /content/drive/MyDrive/FinalProjectDL/final_run/metrics_table.png
  /content/drive/MyDrive/FinalProjectDL/final_run/all_results.pkl

Per-subject figures:

  dog
    main grid:       /content/drive/MyDrive/FinalProjectDL/final_run/dog/main/grid.png
    no_prior grid:   /content/drive/MyDrive/FinalProjectDL/final_run/dog/no_prior/grid.png
    prior_vs_no:     /content/drive/MyDrive/FinalProjectDL/final_run/dog/prior_vs_no_prior.png

  bear_plushie
    main grid:       /content/drive/MyDrive/FinalProjectDL/final_run/bear_plushie/main/grid.png
    no_prior grid:   /content/drive/MyDrive/FinalProjectDL/final_run/bear_plushie/no_prior/grid.png
    prior_vs_no:     /content/drive/MyDrive/FinalProjectDL/final_run/bear_plushie/prior_vs_no_prior.png

  colorful_sneaker
    main grid:       /content/drive/MyDrive/FinalProjectDL/final_run/colorful_sn

# Appendix: Multi-Subject Dog Mixing Experiment
⚠️ **Exploratory — beyond the DreamBooth paper's scope.**

Dogs chosen: `dog` (corgi — compact, red/tan+white, short legs) vs `dog3` (white curly Bichon-type — all-white, curly coat, compact/round body). These differ maximally in color, coat texture, body shape, and breed.

In [ ]:
def make_mixing_instance_sources_shared():
    dog_a, dog_b = CFG["mixing_dogs"]
    prompt = "a photo of a sks dog"
    return [
        (DATASET_ROOT / dog_a, prompt),
        (DATASET_ROOT / dog_b, prompt),
    ]

def make_mixing_instance_sources_distinct():
    dog_a, dog_b = CFG["mixing_dogs"]
    return [
        (DATASET_ROOT / dog_a, "a photo of a sks dog"),
        (DATASET_ROOT / dog_b, "a photo of a zwx dog"),
    ]


In [ ]:
if CFG["RUN_MIXING_APPENDIX"]:
    print("Running exploratory mixing appendix: shared_sks")
    shared_result = run_experiment(
        subject_name="mixing_appendix/shared_sks",
        run_name="shared_sks",
        class_noun="dog", is_living=True,
        n_images=None, with_prior=True, train_text_encoder=False,
        instance_sources=make_mixing_instance_sources_shared(),
    )
    print("✓ Variant 1 (shared_sks) complete")
else:
    print("Mixing appendix disabled (RUN_MIXING_APPENDIX = False)")


Running exploratory mixing appendix: shared_sks


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: shared_sks
  Prior: True, Train TE: False, n_images: all, steps: 800, lr: 3e-06
  step 1: loss=1.4482 inst=0.7504 prior=0.6979
  step 100: loss=0.1119 inst=0.1065 prior=0.0054
  step 200: loss=0.4711 inst=0.2188 prior=0.2523
  step 300: loss=0.0994 inst=0.0292 prior=0.0702
  step 400: loss=0.3974 inst=0.1339 prior=0.2635
  step 500: loss=0.6100 inst=0.1072 prior=0.5028
  step 600: loss=0.1871 inst=0.1599 prior=0.0272
  step 700: loss=0.4427 inst=0.1657 prior=0.2770
  step 800: loss=0.0501 inst=0.0355 prior=0.0147
  Scoring mixing_appendix/shared_sks_shared_sks...


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
  -> appended metrics: mixing_appendix/shared_sks_shared_sks
✓ Variant 1 (shared_sks) complete


In [ ]:
distinct_pipe = None

if CFG["RUN_MIXING_APPENDIX"] and CFG["RUN_MIXING_VARIANT_2"]:
    print("Running exploratory mixing appendix: distinct_tokens")
    distinct_result, distinct_pipe = run_experiment(
        subject_name="mixing_appendix/distinct_tokens",
        run_name="distinct_tokens",
        class_noun="dog", is_living=True,
        n_images=None, with_prior=True, train_text_encoder=False,
        instance_sources=make_mixing_instance_sources_distinct(),
        eval_identifiers=["sks", "zwx"],
        return_pipe=True,
    )
    print("✓ Variant 2 (distinct_tokens) complete")
elif CFG["RUN_MIXING_APPENDIX"]:
    print("Variant 2 disabled (RUN_MIXING_VARIANT_2 = False)")


Running exploratory mixing appendix: distinct_tokens


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]


Training: distinct_tokens
  Prior: True, Train TE: False, n_images: all, steps: 800, lr: 3e-06
  step 1: loss=1.4482 inst=0.7504 prior=0.6979
  step 100: loss=0.1119 inst=0.1064 prior=0.0054
  step 200: loss=0.4711 inst=0.2188 prior=0.2523
  step 300: loss=0.0993 inst=0.0291 prior=0.0702
  step 400: loss=0.3975 inst=0.1339 prior=0.2636
  step 500: loss=0.6095 inst=0.1067 prior=0.5028
  step 600: loss=0.1865 inst=0.1594 prior=0.0271
  step 700: loss=0.4430 inst=0.1658 prior=0.2771
  step 800: loss=0.0496 inst=0.0350 prior=0.0146
  Scoring mixing_appendix/distinct_tokens_distinct_tokens...


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
  -> appended metrics: mixing_appendix/distinct_tokens_distinct_tokens
✓ Variant 2 (distinct_tokens) complete


In [ ]:
if (CFG["RUN_MIXING_APPENDIX"] and CFG["RUN_MIXING_VARIANT_2"]
        and CFG["RUN_MIXING_VARIANT_3"]):
    assert distinct_pipe is not None, "Variant 3 requires Variant 2 trained in this session"
    comp_dir = OUT / "mixing_appendix" / "compositional"
    comp_dir.mkdir(parents=True, exist_ok=True)

    comp_prompts = [
        "a photo of a sks dog and a zwx dog playing",
        "a photo of a sks dog next to a zwx dog",
        "a photo of a sks dog and a zwx dog in the park",
        "a sks dog watching a zwx dog",
    ]

    comp_generations = generate_images(
        distinct_pipe, comp_prompts, comp_dir / "generated",
        images_per_prompt=4, prefix="comp",
    )
    image_rows = []
    for prompt, paths in zip(comp_prompts, comp_generations):
        image_rows.append([Image.open(p) for p in paths])
    make_labeled_grid(image_rows, comp_prompts, comp_dir / "grid.png")

    ALL_RESULTS["mixing_appendix/compositional"] = {
        "run_key": "mixing_compositional",
        "subject": "mixing_appendix",
        "run_name": "compositional",
        "run_dir": str(comp_dir),
        "generations": comp_generations,
        "prompts": comp_prompts,
    }
    save_all_results()
    del distinct_pipe
    free_memory()
    print("✓ Variant 3 (compositional) complete")
elif CFG["RUN_MIXING_APPENDIX"]:
    if distinct_pipe is not None:
        del distinct_pipe
        free_memory()
    print("Variant 3 disabled")


✓ Variant 3 (compositional) complete


In [ ]:
if CFG["RUN_MIXING_APPENDIX"]:
    summary_paths = []
    summary_titles = []

    shared_grid = OUT / "mixing_appendix" / "shared_sks" / "shared_sks" / "grid.png"
    if shared_grid.exists():
        summary_paths.append(shared_grid)
        summary_titles.append("Variant 1: shared sks")

    distinct_grid = OUT / "mixing_appendix" / "distinct_tokens" / "distinct_tokens" / "grid.png"
    if distinct_grid.exists():
        summary_paths.append(distinct_grid)
        summary_titles.append("Variant 2: distinct tokens")

    comp_grid = OUT / "mixing_appendix" / "compositional" / "grid.png"
    if comp_grid.exists():
        summary_paths.append(comp_grid)
        summary_titles.append("Variant 3: compositional")

    if summary_paths:
        stitch_existing_grids(
            summary_paths, summary_titles,
            OUT / "mixing_appendix" / "mixing_summary.png",
        )
    else:
        print("No mixing grids found to summarize.")
else:
    print("Mixing appendix disabled.")


  Saved /content/drive/MyDrive/FinalProjectDL/final_run/mixing_appendix/mixing_summary.png


## Optional final cell: release Colab runtime

In [ ]:
if CFG["RUN_FINAL_UNASSIGN"]:
    from google.colab import runtime
    runtime.unassign()
else:
    print("RUN_FINAL_UNASSIGN is False; runtime kept alive.")
